# 05 — Pseudo-Labeling (Self-Training) — DistilBERT

Iteratively fine-tunes `utils.config.CLASSIFIER_MODEL_NAME` (DistilBERT) on
the 5%-per-class labeled seed (~15/class on master_data.csv, much thinner
than AG News's ~1,500/class), predicts on the full unlabeled pool, and
absorbs high-confidence predictions each round, targeting 98% coverage.
Starts from the AG-News-tuned confidence threshold (0.80) — at this much
thinner 16-way seed the round-0 behavior may differ; if round 0 stalls (0
labels absorbed), Step 2 below re-runs at a lower threshold, following the
same tuning methodology documented previously.

**Result: complete stall, not resolved by retuning.** After Task 5's scope
reduction the labeled seed is only 16 rows total (1 example per class, not
~15/class as first anticipated). At 0.80, 0.50, 0.35, and 0.25 confidence
threshold, round 0 absorbs exactly **0** pseudo-labels every time — a
DistilBERT model fine-tuned on 1 example/class never reaches even a 0.25
softmax confidence on any unlabeled row, so the loop stops immediately at
each threshold with 5.0% coverage (just the seed itself) and the final
model (trained on the 16-row seed only) scores 12.5% test accuracy — barely
above the 6.25% random-guess baseline for 16 classes. Unlike the AG News
story this notebook was adapted from, this is not a threshold-tuning
problem: at 1 example/class there is no self-training signal to absorb at
any reasonable threshold. **Final reported threshold: 0.50** (matching the
brief's own first fallback value) — see `history` and `docs/superpowers/plans/2026-08-15-autolabel-notebooks.md`
"Final Run" note for the underlying labeled-seed-size context.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.metrics import evaluate_label_quality, evaluate_semisupervised
from utils.modeling import get_predictions, pseudo_label_loop
from utils.samples import save_full_output, save_label_samples

In [2]:
labeled_df = pd.read_parquet(config.PROCESSED_DIR / "labeled.parquet")
unlabeled_df = pd.read_parquet(config.PROCESSED_DIR / "unlabeled.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

overlap = set(unlabeled_df["text"]) & set(test_clean["text"])
assert len(overlap) == 0, f"{len(overlap)} rows leaked between train pool and test set"
print(f"Labeled seed: {len(labeled_df)} | Unlabeled pool: {len(unlabeled_df)} | Test: {len(test_clean)}")

Labeled seed: 16 | Unlabeled pool: 303 | Test: 80


In [3]:
CONFIDENCE_THRESHOLD = 0.50  # 0.80 stalled at round 0 (0 new labels); 0.50/0.35/0.25 all stalled identically (see markdown above) — 0.50 kept as the final reported threshold

final_model, final_tokenizer, current_labeled, history = pseudo_label_loop(
    labeled_df, unlabeled_df,
    model_name=config.CLASSIFIER_MODEL_NAME,
    confidence_threshold=CONFIDENCE_THRESHOLD, epochs=3,
    target_coverage=0.98, max_iterations=10)

for h in history:
    print(h)

Total sample: 319 | target coverage: 98% | confidence threshold: 0.5


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


{'iteration': 0, 'total_sample': 319, 'new_labels': 0, 'new_labels_pct_of_total': 0.0, 'labeled_size': 16, 'coverage': 0.050156739811912224, 'unlabeled_size': 303}


In [4]:
# NOTE: `pseudo_only`'s `summary` column (inherited from pseudo_label_loop's
# internal concatenation, not re-selected from unlabeled_df below) has never
# been exercised by an actual absorbed pseudo-label row in any run so far —
# every run at this seed size (1 example/class) absorbed 0 pseudo-labels, so
# `pseudo_only` was always empty. If a future run at a thicker seed absorbs
# pseudo-labels, verify `pseudo_only["summary"]` actually holds correct,
# non-stale values for those rows before trusting this notebook's
# full_labels_pseudo_labeling_train_pool.csv output.
pseudo_only = current_labeled.iloc[len(labeled_df):]
merged = pseudo_only.merge(unlabeled_df[["text", "true_label"]], on="text", how="left")
unresolved = unlabeled_df[~unlabeled_df["text"].isin(pseudo_only["text"])]

label_quality = evaluate_label_quality(
    true_labels=merged["true_label"].to_numpy(),
    pseudo_labels=merged["label"].to_numpy())
print("Pseudo-label quality:", label_quality)

save_label_samples(
    merged["text"], merged["label"].to_numpy(), merged["true_label"].to_numpy(),
    config.CLASS_NAMES, n_per_class=2, seed=config.SEED,
    path=config.RESULTS_DIR / "sample_labels_pseudo_labeling_train_pool.csv")

full_pool_texts = pd.concat([merged["text"], unresolved["text"]], ignore_index=True)
full_pool_predicted = pd.concat(
    [merged["label"], pd.Series(-1, index=unresolved.index)], ignore_index=True)
full_pool_true = pd.concat([merged["true_label"], unresolved["true_label"]], ignore_index=True)
full_pool_summary = pd.concat([merged["summary"], unresolved["summary"]], ignore_index=True)

save_full_output(
    full_pool_texts, full_pool_predicted.to_numpy(), full_pool_true.to_numpy(), config.CLASS_NAMES,
    extra_columns={"summary": full_pool_summary.tolist()},
    path=config.RESULTS_DIR / "full_labels_pseudo_labeling_train_pool.csv")
print("Saved sample + full-row train-pool outputs for pseudo_labeling.")

Pseudo-label quality: {'Label Accuracy': 0.0, 'Label Macro F1': 0.0, 'Coverage': 0.0}
Saved sample + full-row train-pool outputs for pseudo_labeling.


In [5]:
test_probs = get_predictions(final_model, final_tokenizer, test_clean["text"].tolist())
test_preds = test_probs.argmax(axis=1)

semisup_results, report, cm = evaluate_semisupervised(
    test_clean["label"].to_numpy(), test_preds, config.CLASS_NAMES,
    save_path=config.RESULTS_DIR / "confusion_matrix_pseudo_labeling.png")
print(report)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_pseudo_labeling.json", "w") as f:
    json.dump({"test_metrics": semisup_results, "label_quality": label_quality,
               "history": history, "confidence_threshold": CONFIDENCE_THRESHOLD}, f, indent=2)
print("Saved pseudo-labeling results.")

save_label_samples(
    test_clean["text"], test_preds, test_clean["label"].to_numpy(),
    config.CLASS_NAMES, confidence=test_probs.max(axis=1), n_per_class=2, seed=config.SEED,
    path=config.RESULTS_DIR / "sample_labels_pseudo_labeling_test.csv")
save_full_output(
    test_clean["text"], test_preds, test_clean["label"].to_numpy(), config.CLASS_NAMES,
    confidence=test_probs.max(axis=1), extra_columns={"summary": test_clean["summary"].tolist()},
    path=config.RESULTS_DIR / "full_labels_pseudo_labeling_test.csv")
print("Saved sample + full-row test outputs for pseudo_labeling.")

C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0

                precision    recall  f1-score   support

ARTS & CULTURE       0.00      0.00      0.00         5
      BUSINESS       0.00      0.00      0.00         5
        COMEDY       0.00      0.00      0.00         5
         CRIME       0.75      0.60      0.67         5
     EDUCATION       0.00      0.00      0.00         5
 ENTERTAINMENT       0.00      0.00      0.00         5
   ENVIRONMENT       0.00      0.00      0.00         5
        HEALTH       0.00      0.00      0.00         5
         MEDIA       0.00      0.00      0.00         5
          NEWS       0.00      0.00      0.00         5
      POLITICS       0.12      0.40      0.18         5
      RELIGION       0.00      0.00      0.00         5
       SCIENCE       0.00      0.00      0.00         5
        SPORTS       0.00      0.00      0.00         5
          TECH       0.00      0.00      0.00         5
         WOMEN       0.09      1.00      0.16         5

      accuracy                           0.12 